# STT 09 — Le grand modèle sur l'ancien dev et l'ancien test, et le RTF sur CPU

**But :**
1. **La vraie comparaison** : évaluer `best.pt` (grand dataset, époque 19) sur **les mêmes clips** que les anciens modèles. Les 72,5 % (référence) et 70,8 % (SpecAugment) étaient mesurés sur l'ancien dev ; le 37,4 % sur le dev officiel. Ce ne sont pas les mêmes examens.
2. **Le RTF sur CPU** (Real-Time Factor) : temps de calcul ÷ durée de l'audio. Au-dessus de 1, le modèle est trop lent pour le direct.

**Exécuté en local** (CPU, 4 cœurs) : les données du petit dataset et `best.pt` sont déjà sur le PC.

**Recouvrement (journal, EDA point 7b) :** 0 clip de l'ancien dev dans le grand train ; **1 clip** de l'ancien test dans le grand train, gardé faute de la liste du grand train sur le PC (effet < 0,5 point sur 291 clips).

**Règles de données (conditions MDC) :** aucune phrase n'est affichée, seulement des chiffres.

## Cellule 1 — L'ancien dev et l'ancien test

In [1]:
import time, json
import torch
from torch.utils.data import DataLoader
from stt.dataset import load_split, CommonVoiceDataset, collate_batch

torch.set_num_threads(4)                      # le CPU a 4 cœurs : on les utilise tous
FOLDERS = ["data/fr-FR", "data/fr-CA"]        # le petit dataset, exactement comme à l'entraînement d'avant

tables = {s: load_split(FOLDERS, s) for s in ("dev", "test")}   # même lecture et même filtre qu'avant
for s, t in tables.items():
    print(f"Ancien {s} : {len(t):,} clips gardés par load_split")

loaders = {s: DataLoader(CommonVoiceDataset(t, augment=False),  # augment=False : pas de SpecAugment en évaluation
                         batch_size=32, shuffle=False,          # shuffle=False : ordre fixe, résultat reproductible
                         collate_fn=collate_batch)
           for s, t in tables.items()}

Ancien dev : 484 clips gardés par load_split
Ancien test : 285 clips gardés par load_split


## Cellule 2 — Le meilleur modèle du grand dataset

`map_location` ramène sur CPU des poids sauvegardés sur GPU. `model.eval()` coupe le dropout : aucun apprentissage ici.

In [2]:
from stt.model import SpeechTransformer

device = "cpu"                                                    # en local, pas de GPU
ckpt = torch.load("runs/grand_s_specaug/best.pt", map_location=device,   # map_location : les poids ont été
                  weights_only=False)                                    # sauvegardés sur GPU, on les ramène sur CPU
model = SpeechTransformer().to(device)                            # taille S, comme à l'entraînement
model.load_state_dict(ckpt["model"])                              # on remplace les poids au hasard par les poids appris
model.eval()                                                      # dropout coupé : on mesure le vrai modèle
print(f"Modèle chargé : step {ckpt['step']:,} | WER dev officiel {ckpt['wer']:.2f} %")

Modèle chargé : step 363,508 | WER dev officiel 37.38 %


## Cellule 3 — WER, CER et loss sur l'ancien dev et l'ancien test

Même calcul que `stt_08`, dans une fonction. On additionne les erreurs au fur et à mesure : aucune phrase n'est gardée. **À comparer :** ancien dev contre 70,8 % (meilleur ancien modèle, mêmes 484 clips).

In [3]:
from stt.metrics import greedy_decode, edit_distance
from stt.text import decode

ctc_loss = torch.nn.CTCLoss(blank=0, zero_infinity=True)

def evaluate_numbers(loader):
    # Renvoie loss, WER, CER et part de clips parfaits. Aucune phrase n'est gardée.
    word_err = words = char_err = chars = perfect = n_clips = 0
    total_loss, n_batches = 0.0, 0
    with torch.no_grad():                                        # pas de gradients : évaluation seulement
        for batch in loader:
            features, feature_lengths, targets, target_lengths = batch
            log_probs, out_lengths = model(features, feature_lengths)
            total_loss += ctc_loss(log_probs.transpose(0, 1), targets, out_lengths, target_lengths).item()
            n_batches += 1
            for b, hyp in enumerate(greedy_decode(log_probs, out_lengths)):
                ref = decode(targets[b, :target_lengths[b]].tolist())   # ref et hyp ne sont jamais affichés
                we = edit_distance(ref.split(), hyp.split())
                word_err += we;  words += len(ref.split())
                char_err += edit_distance(list(ref), list(hyp));  chars += len(ref)
                perfect += (we == 0);  n_clips += 1
    return {"clips": n_clips, "loss": round(total_loss / n_batches, 3),
            "wer": round(100 * word_err / words, 2),             # total des erreurs ÷ total des mots, comme error_rate
            "cer": round(100 * char_err / chars, 2),
            "perfect_clips_pct": round(100 * perfect / n_clips, 1)}

results = {}
for s, loader in loaders.items():
    t0 = time.time()
    results[s] = evaluate_numbers(loader)
    r = results[s]
    print(f"Ancien {s:4s} | {r['clips']} clips | WER {r['wer']:.2f} % | CER {r['cer']:.2f} % | "
          f"loss {r['loss']:.3f} | clips parfaits {r['perfect_clips_pct']:.1f} % | {time.time() - t0:.0f} s")


Ancien dev  | 484 clips | WER 38.25 % | CER 11.53 % | loss 0.416 | clips parfaits 5.8 % | 56 s


Ancien test | 285 clips | WER 44.65 % | CER 14.52 % | loss 0.528 | clips parfaits 3.2 % | 52 s


## Cellule 4 — RTF sur CPU, un clip à la fois (batch de 1, comme sur le robot)

Chronomètre : log-Mel + modèle + décodage. La lecture du mp3 n'est pas comptée (le micro du robot donne des échantillons bruts). `time.perf_counter()` : horloge précise ; pas de `torch.cuda.synchronize()` sur CPU. 5 clips d'échauffement non comptés. **Limite :** CPU du PC, pas celui du robot.

In [4]:
from stt.audio import load_audio, log_mel, TARGET_SR

compute, durations = [], []
with torch.no_grad():
    for i, path in enumerate(tables["test"]["audio_path"]):
        waveform = load_audio(path)                     # lecture du mp3 : PAS chronométrée (sur le robot, le micro
                                                        # donne directement les échantillons, il n'y a pas de mp3)
        t0 = time.perf_counter()                        # début du chronomètre
        features = log_mel(waveform).unsqueeze(0)       # (1, instants, 80) : batch de 1, comme sur le robot
        lengths = torch.tensor([features.shape[1]])
        log_probs, out_lengths = model(features, lengths)
        greedy_decode(log_probs, out_lengths)           # le texte n'est pas gardé : seul le temps compte
        elapsed = time.perf_counter() - t0              # fin du chronomètre
        if i >= 5:                                      # les 5 premiers clips servent d'échauffement
            compute.append(elapsed)                     # (le premier appel de PyTorch est toujours plus lent)
            durations.append(len(waveform) / TARGET_SR) # durée de l'audio en secondes

compute, durations = torch.tensor(compute), torch.tensor(durations)
rtf = compute / durations                               # RTF de chaque clip
print(f"{len(rtf)} clips | RTF global {compute.sum() / durations.sum():.3f} | "
      f"RTF médian {rtf.median():.3f} | pire RTF {rtf.max():.3f}")
print(f"Temps de calcul par clip : médian {1000 * compute.median():.0f} ms | max {1000 * compute.max():.0f} ms "
      f"(pour des clips de {durations.median():.1f} s en médiane)")


280 clips | RTF global 0.014 | RTF médian 0.013 | pire RTF 0.122
Temps de calcul par clip : médian 75 ms | max 792 ms (pour des clips de 6.0 s en médiane)


## Cellule 5 — Sauvegarde des chiffres

Dans `runs/grand_s_specaug/comparaison_ancien_dev_test.json` (hors du repo). Uniquement des chiffres.

In [5]:
summary = {"checkpoint": {"path": "runs/grand_s_specaug/best.pt", "step": ckpt["step"],
                          "dev_officiel_wer": round(ckpt["wer"], 2)},
           "ancien_dev": results["dev"], "ancien_test": results["test"],
           "ancien_meilleur_modele_wer_ancien_dev": 70.8,          # SpecAugment, petit dataset (journal du 02/11)
           "rtf_cpu": {"cpu": "Intel i5-7300U, 4 threads", "clips": len(rtf),
                       "rtf_global": round(float(compute.sum() / durations.sum()), 4),
                       "rtf_median": round(float(rtf.median()), 4), "rtf_max": round(float(rtf.max()), 4),
                       "latence_mediane_ms": round(1000 * float(compute.median())),
                       "latence_max_ms": round(1000 * float(compute.max()))}}
with open("runs/grand_s_specaug/comparaison_ancien_dev_test.json", "w") as f:
    json.dump(summary, f, indent=1)                    # dans runs/ : hors du repo, comme test_result.json
print(json.dumps(summary, indent=1))


{
 "checkpoint": {
  "path": "runs/grand_s_specaug/best.pt",
  "step": 363508,
  "dev_officiel_wer": 37.38
 },
 "ancien_dev": {
  "clips": 484,
  "loss": 0.416,
  "wer": 38.25,
  "cer": 11.53,
  "perfect_clips_pct": 5.8
 },
 "ancien_test": {
  "clips": 285,
  "loss": 0.528,
  "wer": 44.65,
  "cer": 14.52,
  "perfect_clips_pct": 3.2
 },
 "ancien_meilleur_modele_wer_ancien_dev": 70.8,
 "rtf_cpu": {
  "cpu": "Intel i5-7300U, 4 threads",
  "clips": 280,
  "rtf_global": 0.0144,
  "rtf_median": 0.0128,
  "rtf_max": 0.1217,
  "latence_mediane_ms": 75,
  "latence_max_ms": 792
 }
}
